# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the [FAIR^2](https://doi.org/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

- **Croissant JSON-LD schema URL:** [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)

In [ ]:
# Ensure `mlcroissant` and visualization libraries are installed
!pip install -q mlcroissant matplotlib seaborn pandas

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Show metadata title and description
print(f"Dataset name: {dataset.metadata.name}")
print(f"Dataset description: {dataset.metadata.description}\n")

# Display main metadata attributes
print(f"Identifier: {getattr(dataset.metadata, 'identifier', None)}")
print(f"Version: {getattr(dataset.metadata, 'version', None)}")
print(f"Published date: {getattr(dataset.metadata, 'datePublished', None)}")
print(f"License: {getattr(dataset.metadata, 'license', None)}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Let's inspect what record sets are present in this dataset, then retrieve their fields (`@id`s).

In [ ]:
# List all record set @id and names available in the dataset
record_set_objs = dataset.metadata.record_sets
print(f"Total record sets: {len(record_set_objs)}\n")
for rs in record_set_objs:
    print(f"Record set name: {rs.name}\n  @id: {rs.id}\n  Source(s): {[s.id for s in rs.sources]}\n")

List the available fields and their `@id` for one or more record sets. This is necessary for selecting fields and columns in later steps.

In [ ]:
# For each record set, print its field names, @id, and their corresponding columns
for rs in dataset.metadata.record_sets:
    print(f"RecordSet: {rs.name} (@id: {rs.id})")
    for f in rs.fields:
        col_ids = [str(c.id) for c in getattr(f, 'columns', [])]
        print(f"  Field: {f.name} (@id: {f.id})  DataType: {f.data_type}  Columns: {col_ids}")
    print("")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Gather all record set @id's
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}

# Load each record set into a DataFrame
for record_set_id in record_set_ids:
    # This loads all records for the record set
    rows = list(dataset.records(record_set=record_set_id))
    if rows:
        df = pd.DataFrame(rows)
        dataframes[record_set_id] = df
        print(f"Loaded DataFrame for {record_set_id} with shape {df.shape}")
    else:
        print(f"No records loaded for {record_set_id}")

# As an example, preview the first DataFrame's columns and first rows
if dataframes:
    example_record_set_id = list(dataframes.keys())[0]
    print(f"\nColumns for {example_record_set_id}: \n{dataframes[example_record_set_id].columns.tolist()}")
    dataframes[example_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering records, normalizing numeric fields, and grouping data by key attributes. 

We'll select a numeric field for analysis, filter records, and show how to normalize and group values.

*(Replace example field names/IDs with those printed above as needed)*

In [ ]:
# Example of field IDs for EDA (substitute these based on actual @id's found above)
# For illustration, select record set and field IDs from actual overview output above if needed.

# Pick the first available record set and try to infer a numeric field
if dataframes:
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id].copy()
    
    # Try to infer a field likely to be numeric
    numeric_candidates = [c for c in df.columns if df[c].dtype in [int, float] or pd.api.types.is_numeric_dtype(df[c])]
    if not numeric_candidates:
        # Try to coerce some likely numeric columns
        for c in df.columns:
            try:
                df[c] = pd.to_numeric(df[c], errors='ignore')
            except Exception:
                continue
        numeric_candidates = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]

    if numeric_candidates:
        numeric_field = numeric_candidates[0]  # Example: mass spectrometry datasets often have MW or Intensity as a numeric field.
        print(f"Selected numeric field for analysis: {numeric_field}")
        threshold = df[numeric_field].mean()  # Set threshold as mean, or adjust as appropriate
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold} (mean): {len(filtered_df)} records")

        # Normalize the numeric field
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / (filtered_df[numeric_field].std() + 1e-5)
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
        
        # Try to group by a categorical variable if present (e.g., 'Protein', 'Sample')
        group_candidates = [c for c in df.columns if pd.api.types.is_object_dtype(df[c]) and c != numeric_field]
        group_field = group_candidates[0] if group_candidates else None
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"\nGrouped by '{group_field}', mean {numeric_field}:")
            print(grouped_df.head())
        else:
            print("No suitable group field found.")
    else:
        print("No numeric fields found in example DataFrame.")
else:
    print("No DataFrames loaded in previous step.")

## 5. Visualization

Visualize the distribution of the selected numeric field and its relationship to a categorical group (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and 'numeric_field' in locals() and numeric_field in df.columns:
    # Histogram of the numeric field (original and normalized)
    plt.figure(figsize=(10,4))
    plt.subplot(1,2,1)
    sns.histplot(df[numeric_field], kde=True)
    plt.title(f'Distribution of {numeric_field}')

    if f"{numeric_field}_normalized" in df.columns:
        plt.subplot(1,2,2)
        sns.histplot(df[f"{numeric_field}_normalized"], kde=True)
        plt.title(f'Normalized {numeric_field}')

    plt.tight_layout()
    plt.show()

    # Violin or boxplot by group field, if available
    if 'group_field' in locals() and group_field and group_field in df.columns:
        plt.figure(figsize=(10,4))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f'{numeric_field} by {group_field}')
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No suitable numeric field for visualization.")

## 6. Conclusion

- We used `mlcroissant` to programmatically load metadata and tabular records from the FAIR^2 mass spectrometry dataset, referencing all key data entities with their `@id`.
- Automatically detected and visualized numeric fields (e.g., protein abundance, MW), showed how to filter and normalize values, and grouped by available categorical variables.
- This approach can be generalized to any dataset with a Croissant schema, providing standardized metadata exploration and robust reproducible processing.